# Stage 1: source manifest

**What this stage does:** writes down exactly which raw data we used.

We unzip the Cricsheet archive (only the first time) and make a list of every match file with its size,
SHA-256 checksum and Cricsheet schema version. With this list anyone can check that they are working with the
same files as us.

**Reads:** `data/raw/england_male_json.zip`
**Writes:** `data/interim/source_manifest.csv` and `data/interim/source_summary.json`

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_01_data_loading.ipynb
stage_log = get_stage_logger("01_manifest")

In [2]:
match_folder = extract_archive_if_needed()
source_manifest = build_source_manifest(match_folder)
source_manifest.to_csv(INTERIM_FOLDER / "source_manifest.csv", index=False)
source_manifest.head()

,filename,bytes,sha256,readable,data_version,created,revision
0,1022347.json,201990,d373514b89cd5d1f8a52a78b10260bc56130f127c96b3c...,True,1.3.0,2017-06-02,2
1,1022357.json,198851,a8fa397b85da5d9cb42e3847d2c2050b35ea1a38f0a982...,True,1.3.0,2017-06-07,2
2,1022365.json,186899,f7c9a63887dda30980ebc6f1226f922b97baebbaf382d5...,True,1.3.0,2017-06-11,2
3,1022371.json,183241,979a6cacbca7d843ac4cdae01727a751f707a615c4031d...,True,1.3.0,2017-06-19,2
4,1027315.json,206272,987fdc11c1e138a541f20b919a224919807a122942ad1d...,True,1.3.0,2017-04-02,2


A short summary of the whole archive. We did not write down the day we downloaded the zip, so we keep the
file's own date instead.

In [3]:
archive_file_info = RAW_ARCHIVE_PATH.stat()
source_summary = {
    "source_url": ARCHIVE_SOURCE_URL,
    "archive": RAW_ARCHIVE_PATH.name,
    "archive_bytes": archive_file_info.st_size,
    "archive_sha256": file_checksum(RAW_ARCHIVE_PATH),
    "archive_file_modified_utc": datetime.fromtimestamp(archive_file_info.st_mtime, timezone.utc).isoformat(timespec="seconds"),
    "json_files": int(len(source_manifest)),
    "unreadable_files": int((~source_manifest["readable"]).sum()),
    "data_versions": source_manifest["data_version"].value_counts().to_dict(),
    # The archive also holds a README and a licence file; we list them so it's clear they were seen.
    "other_files": sorted(path.name for path in match_folder.iterdir() if path.suffix != ".json"),
}
(INTERIM_FOLDER / "source_summary.json").write_text(json.dumps(source_summary, indent=2))
stage_log.info("manifest: %s", source_summary)

2026-11-19 18:04:59,092 INFO partnerships.01_manifest: manifest: {'source_url': 'https://cricsheet.org/downloads/england_male_json.zip', 'archive': 'england_male_json.zip', 'archive_bytes': 9271512, 'archive_sha256': '531b679f0cf9e50496e8b4d7015dbfca04dcd4d1d37efa51ff6f4d51dbeb004d', 'archive_file_modified_utc': '2026-11-11T16:42:01+00:00', 'json_files': 961, 'unreadable_files': 0, 'data_versions': {'1.3.0': 961}, 'other_files': ['LICENSE.txt', 'README.txt']}
